# Nhận diện ảnh Flowers bên ngoài dataset bằng hai model

Workflow này dùng VGG-16 (CNN thuần) và RepLKNet-31B để nhận diện một hoặc nhiều ảnh hoa hoàn toàn bên ngoài dataset. Nếu checkpoint đầy đủ đã có, notebook dùng checkpoint đó; nếu chưa có, notebook tự train nhẹ một epoch. Ảnh bên ngoài chỉ được dùng ở bước inference, không được đưa vào train, validation, test hoặc chọn checkpoint.

## Thiết kế chống data leakage

Dataset nội bộ vẫn dùng split manifest cố định. Ảnh mới phải khác `data/flowers/`. Notebook kiểm tra đường dẫn, manifest và SHA-256 của toàn bộ ảnh dataset trước khi cho inference, nên cả trường hợp sao chép ảnh rồi đổi tên cũng bị chặn. Vì ảnh ngoài không có nhãn kiểm chứng trong dataset, đầu ra của nó được báo cáo dưới dạng dự đoán và confidence, không cộng vào Accuracy hoặc Macro-F1.

`LIGHT_EPOCHS = 1` chỉ là phương án dự phòng khi chưa có checkpoint đầy đủ; kết luận khoa học vẫn lấy từ run nhiều epoch và nhiều seed trong `Flower_Experiment_Vietnamese.ipynb`.

In [ ]:
import hashlib
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch 
from PIL import Image # Thư viện Pillow để xử lý ảnh

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / 'src' / 'flower_experiment.py').exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError('Không tìm thấy project root RepLKNet.')

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
DATA_DIR = PROJECT_ROOT / 'data' / 'flowers'
LIGHT_RESULTS_DIR = PROJECT_ROOT / 'results' / 'flowers_light_external_b4'
EXTERNAL_DIR = PROJECT_ROOT / 'external_images'
# Có thể đặt một file hoặc cả thư mục ảnh hoa bên ngoài dataset.
# Hoặc đặt biến môi trường REPLKNET_EXTERNAL_IMAGE trỏ tới file/thư mục đó.
EXTERNAL_IMAGE_PATH = Path(os.environ.get(
    'REPLKNET_EXTERNAL_IMAGE',
    str(EXTERNAL_DIR / 'daisy_flower.jpg'),
))
LIGHT_EPOCHS = 1
LIGHT_BATCH_SIZE = 4
LIGHT_SEED = 42
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    raise RuntimeError('CUDA không khả dụng. Hãy chọn kernel Python (RepLKNet Demo).')

print(f'Project root: {PROJECT_ROOT}')
print(f'Python: {sys.executable}')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'File/thư mục external dự kiến: {EXTERNAL_IMAGE_PATH}')

## 1. Kiểm tra ảnh external trước khi train/inference

Đặt một ảnh hoặc cả thư mục ảnh hoa ngoài dataset vào `external_images/`, hoặc sửa `EXTERNAL_IMAGE_PATH`/biến môi trường `REPLKNET_EXTERNAL_IMAGE` thành đường dẫn file hoặc thư mục cần nhận diện. Notebook sẽ quét các định dạng ảnh được hỗ trợ, kiểm tra từng ảnh không thuộc train, validation, test và không trùng file ảnh trong dataset. Không đặt ảnh vào `data/flowers/`.

In [ ]:
from torchvision.datasets import ImageFolder
from flower_experiment import build_transforms, locate_imagefolder_root, run_experiment

DATA_ROOT = locate_imagefolder_root(DATA_DIR)
_, EVAL_TRANSFORM = build_transforms(image_size=224)
DATASET = ImageFolder(DATA_ROOT, transform=EVAL_TRANSFORM)
MANIFEST_PATH = PROJECT_ROOT / 'results' / 'flowers' / 'split_manifest.json'
MANIFEST = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
MANIFEST_FILES = set(MANIFEST['files'])

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

SUPPORTED_IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
EXTERNAL_IMAGE_PATH = EXTERNAL_IMAGE_PATH.expanduser().resolve()
EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)
EXTERNAL_IMAGE_PATHS = []

if not EXTERNAL_IMAGE_PATH.exists():
    print('Chưa có file/thư mục external. Đặt ảnh tại:')
    print(f'  {EXTERNAL_IMAGE_PATH}')
    print('Notebook vẫn có thể train nhẹ; cell inference sẽ chờ ảnh external.')
elif EXTERNAL_IMAGE_PATH.is_file():
    candidate_paths = [EXTERNAL_IMAGE_PATH]
elif EXTERNAL_IMAGE_PATH.is_dir():
    candidate_paths = sorted(
        path for path in EXTERNAL_IMAGE_PATH.rglob('*')
        if path.is_file() and path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS
    )
else:
    raise ValueError(f'Đường dẫn external không hợp lệ: {EXTERNAL_IMAGE_PATH}')

if EXTERNAL_IMAGE_PATH.exists() and not candidate_paths:
    raise ValueError(f'Không tìm thấy ảnh được hỗ trợ trong: {EXTERNAL_IMAGE_PATH}')

if EXTERNAL_IMAGE_PATH.exists():
    dataset_hashes = {}
    for dataset_path in DATA_ROOT.rglob('*'):
        if dataset_path.is_file() and dataset_path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS:
            dataset_hashes.setdefault(sha256_file(dataset_path), []).append(dataset_path)

    for external_path in candidate_paths:
        external_path = external_path.expanduser().resolve()
        if DATA_ROOT in external_path.parents:
            raise RuntimeError(
                f'Ảnh external nằm trong data/flowers: {external_path}; dừng để tránh data leakage.'
            )
        try:
            relative_name = str(external_path.relative_to(DATA_ROOT)).replace('\\', '/')
        except ValueError:
            relative_name = None
        if relative_name in MANIFEST_FILES:
            raise RuntimeError(f'Ảnh external xuất hiện trong split manifest: {external_path}')

        external_hash = sha256_file(external_path)
        duplicate_files = dataset_hashes.get(external_hash, [])
        if duplicate_files:
            examples = ', '.join(str(path.relative_to(DATA_ROOT)) for path in duplicate_files[:3])
            raise RuntimeError(
                'Ảnh external trùng nội dung với ảnh trong dataset; dừng để tránh data leakage. '
                f'File trùng: {examples}'
            )
        with Image.open(external_path) as checked_image:
            checked_image.verify()
        EXTERNAL_IMAGE_PATHS.append(external_path)
        print(f'Ảnh external hợp lệ: {external_path.name} | SHA-256: {external_hash}')

print(f'Dataset nội bộ: {len(DATASET)} ảnh | classes={DATASET.classes}')
print(f'Số ảnh external hợp lệ: {len(EXTERNAL_IMAGE_PATHS)}')
print('Ảnh external được dùng cho train: False')
print('Ảnh external chỉ được dùng sau khi hai model đã train xong; không tính vào metrics.')

## 2. Hiển thị ngẫu nhiên một số ảnh hoa trong train split

Cell này chỉ để quan sát nhanh các lớp hoa. Ảnh được lấy từ train split, không dùng để cập nhật trọng số, chọn checkpoint hoặc đánh giá ảnh external.

In [ ]:
import math
import random

RANDOM_IMAGE_SEED = 20260905
N_RANDOM_FLOWER_IMAGES = 8
rng = random.Random(RANDOM_IMAGE_SEED)
TRAIN_INDICES = [int(index) for index in MANIFEST['indices']['train']]

samples_by_class = {class_name: [] for class_name in DATASET.classes}
for index in TRAIN_INDICES:
    sample_path, target = DATASET.samples[index]
    samples_by_class[DATASET.classes[target]].append((Path(sample_path), target))

# Bảo đảm mỗi lớp xuất hiện ít nhất một lần, sau đó bổ sung ảnh ngẫu nhiên.
selected_samples = [
    rng.choice(samples_by_class[class_name])
    for class_name in DATASET.classes
]
all_train_samples = [sample for values in samples_by_class.values() for sample in values]
selected_paths = {sample_path for sample_path, _ in selected_samples}
remaining_samples = [sample for sample in all_train_samples if sample[0] not in selected_paths]
extra_count = max(0, min(N_RANDOM_FLOWER_IMAGES, len(all_train_samples)) - len(selected_samples))
selected_samples.extend(rng.sample(remaining_samples, extra_count))

ncols = 4
nrows = math.ceil(len(selected_samples) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
axes = np.atleast_1d(axes).ravel()
for axis, (sample_path, target) in zip(axes, selected_samples):
    with Image.open(sample_path) as source_image:
        axis.imshow(source_image.convert('RGB'))
    axis.set_title(f'{DATASET.classes[target]}\n{sample_path.name[:24]}')
    axis.axis('off')
for axis in axes[len(selected_samples):]:
    axis.axis('off')
fig.suptitle('Ảnh hoa ngẫu nhiên từ train split - chỉ trực quan hóa', fontsize=16)
plt.tight_layout()
plt.show()
print(f'Đã hiển thị {len(selected_samples)} ảnh; tất cả đều thuộc train split và không dùng cho inference external.')

## 3. Chọn checkpoint train trên Flowers

Hai model chỉ nhìn thấy ảnh trong `data/flowers/`. Notebook ưu tiên checkpoint đã train trong `results/flowers/seed_42/`; nếu checkpoint này không tồn tại, notebook mới chạy light training và lưu riêng vào `results/flowers_light_external_b4/`. Ảnh external chưa được đọc ở bước train.

In [ ]:
FULL_RESULTS_DIR = PROJECT_ROOT / 'results' / 'flowers'
FULL_SEED = 42
FULL_SEED_DIR = FULL_RESULTS_DIR / f'seed_{FULL_SEED}'
FULL_VGG_CHECKPOINT = FULL_SEED_DIR / 'vgg16_best.pt'
FULL_REPLK_CHECKPOINT = FULL_SEED_DIR / 'replknet31b_best.pt'

if FULL_VGG_CHECKPOINT.exists() and FULL_REPLK_CHECKPOINT.exists():
    INFERENCE_RESULTS_DIR = FULL_RESULTS_DIR
    INFERENCE_SEED = FULL_SEED
    print(f'Dùng checkpoint đầy đủ đã train tại {FULL_SEED_DIR}.')
else:
    LIGHT_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    light_seed_dir = LIGHT_RESULTS_DIR / f'seed_{LIGHT_SEED}'
    vgg_done = (light_seed_dir / 'vgg16_metrics.json').exists()
    replk_done = (light_seed_dir / 'replknet31b_metrics.json').exists()
    if not (vgg_done and replk_done):
        light_summary = run_experiment(
            data_dir=DATA_DIR,
            project_root=PROJECT_ROOT,
            output_dir=LIGHT_RESULTS_DIR,
            replk_checkpoint=PROJECT_ROOT / 'weights' / 'RepLKNet-31B_ImageNet-1K_224.pth',
            epochs=LIGHT_EPOCHS,
            batch_size=LIGHT_BATCH_SIZE,
            seeds=[LIGHT_SEED],
            num_workers=0,
            device_name='cuda',
            deterministic=True,
        )
        print(f'Đã hoàn tất light training: {len(light_summary)} model-seed results')
    else:
        print(f'Đã có light run tại {LIGHT_RESULTS_DIR}; giữ nguyên checkpoint hiện có.')
    INFERENCE_RESULTS_DIR = LIGHT_RESULTS_DIR
    INFERENCE_SEED = LIGHT_SEED

INFERENCE_SEED_DIR = INFERENCE_RESULTS_DIR / f'seed_{INFERENCE_SEED}'
print(f'Thư mục checkpoint dùng cho inference: {INFERENCE_SEED_DIR}')

## 4. Nạp hai checkpoint và nhận diện ảnh mới

Classifier vẫn chỉ có 5 lớp Flowers. Nếu ảnh không phải hoa hoặc thuộc loại ngoài 5 lớp, confidence cao không đồng nghĩa với dự đoán đúng; đó là giới hạn open-set của bài toán này.

In [ ]:
from torchvision.models import vgg16
import torch.nn as nn
from replknet_demo import build_replknet, load_checkpoint_flexible

CLASS_NAMES = DATASET.classes
VGG_CHECKPOINT = INFERENCE_SEED_DIR / 'vgg16_best.pt'
REPLK_CHECKPOINT = INFERENCE_SEED_DIR / 'replknet31b_best.pt'

vgg_model = vgg16(weights=None)
vgg_model.classifier[6] = nn.Linear(vgg_model.classifier[6].in_features, len(CLASS_NAMES))
load_checkpoint_flexible(vgg_model, VGG_CHECKPOINT)
vgg_model = vgg_model.cpu().eval()

replk_model, _ = build_replknet(
    PROJECT_ROOT,
    model_name='RepLKNet-31B',
    checkpoint_path=None,
    device=torch.device('cpu'),
    num_classes=len(CLASS_NAMES),
    merge_for_inference=False,
)
load_checkpoint_flexible(replk_model, REPLK_CHECKPOINT)
replk_model.structural_reparam()
replk_model = replk_model.cpu().eval()
MODELS = {'VGG-16 (CNN thuần)': vgg_model, 'RepLKNet-31B': replk_model}

In [ ]:
if not EXTERNAL_IMAGE_PATHS:
    print(f'Chưa nhận diện: chưa có ảnh trong {EXTERNAL_IMAGE_PATH}')
else:
    external_predictions = []
    for external_path in EXTERNAL_IMAGE_PATHS:
        with Image.open(external_path) as source_image:
            external_image = source_image.convert('RGB')
        external_batch = EVAL_TRANSFORM(external_image).unsqueeze(0)
        image_result = {'image_path': external_path, 'predictions': {}}
        for model_name, model in MODELS.items():
            model = model.to(DEVICE).eval()
            with torch.inference_mode():
                probabilities = torch.softmax(model(external_batch.to(DEVICE)), dim=1)[0].cpu()
            model.cpu()
            torch.cuda.empty_cache()
            top_index = int(probabilities.argmax())
            image_result['predictions'][model_name] = {
                'predicted_class': CLASS_NAMES[top_index],
                'confidence': float(probabilities[top_index]),
                'probabilities': probabilities.numpy(),
            }
        external_predictions.append(image_result)

    print(f'Đã nhận diện {len(external_predictions)} ảnh external:')
    for image_result in external_predictions:
        print(f"\n{image_result['image_path'].name}")
        for model_name, result in image_result['predictions'].items():
            print(f"  {model_name}: {result['predicted_class']} ({result['confidence']:.2%})")

    MAX_EXTERNAL_IMAGES_TO_PLOT = 12
    plotted_results = external_predictions[:MAX_EXTERNAL_IMAGES_TO_PLOT]
    x = np.arange(len(CLASS_NAMES))
    fig, axes = plt.subplots(len(plotted_results), 3, figsize=(18, max(4, 4 * len(plotted_results))), squeeze=False)
    for row_index, image_result in enumerate(plotted_results):
        image_path = image_result['image_path']
        with Image.open(image_path) as source_image:
            axes[row_index, 0].imshow(source_image.convert('RGB'))
        axes[row_index, 0].set_title(image_path.name)
        axes[row_index, 0].axis('off')
        for column_index, (model_name, result) in enumerate(image_result['predictions'].items(), start=1):
            axes[row_index, column_index].bar(x, result['probabilities'])
            axes[row_index, column_index].set_xticks(x, CLASS_NAMES, rotation=25, ha='right')
            axes[row_index, column_index].set_ylim(0, 1)
            axes[row_index, column_index].set_ylabel('Xác suất')
            axes[row_index, column_index].set_title(f"{model_name}\n{result['predicted_class']} ({result['confidence']:.2%})")
    fig.suptitle('Dự đoán trên ảnh external - không dùng cho metrics', fontsize=16)
    plt.tight_layout()
    plt.show()
    if len(external_predictions) > MAX_EXTERNAL_IMAGES_TO_PLOT:
        print(f'Chỉ vẽ {MAX_EXTERNAL_IMAGES_TO_PLOT} ảnh đầu; đã nhận diện đủ {len(external_predictions)} ảnh.')

## Ghi nhận để báo cáo

Ảnh external được dùng như một phép kiểm tra ngoài mẫu (external sanity check). Không dùng kết quả của ảnh này để cập nhật trọng số, chọn epoch tốt nhất, điều chỉnh threshold hoặc tính metrics test. Nếu biết nhãn thật của ảnh, có thể ghi nhận đúng/sai riêng trong phụ lục, nhưng không gộp vào test set.